# Mise en place de la VM Monitoring & Analytics

Ce notebook crée la **VM EC2** de l'architecture (`doc/G10-architecture-V2.md`) : le **Scenario Analyzer** et **Grafana OSS** y tournent dans Docker, avec le même `AWS/Grafana/docker-compose.local.yml` qu'en local. Pour l'instant, l'Analyzer lit le **rejeu de fichiers** (`batches.jsonl`). La lecture de DynamoDB viendra quand l'ingestion existera.

Grafana n'est **pas exposé sur Internet** : la VM n'a aucun port entrant, et on y accède par un tunnel **SSM Session Manager**. Il n'y a ni SSH ni clé.

Chaque cellule peut être relancée sans risque. L'arrêt (10) et le nettoyage (11) sont désactivés par défaut.

**Prérequis** : identifiants AWS configurés (`aws configure`) avec les droits EC2, IAM, S3 et SSM ; jeu de rejeu généré (voir `AWS/Analyser/README.md`) ; pour le tunnel, l'AWS CLI et le plugin Session Manager (`brew install awscli && brew install --cask session-manager-plugin`).

## 0. Configuration

À adapter si besoin. `PREFIX` sert à nommer toutes les ressources : le rôle, le security group, l'instance et le bucket, auquel s'ajoute l'identifiant du compte.

In [ ]:
import json
import shutil
import sys
from pathlib import Path

import boto3

try:
    import vm_setup  # noqa: F401
except ModuleNotFoundError:  # iCloud peut masquer le .pth de l'installation éditable
    sys.path.insert(0, str(Path.cwd() / "src"))
from vm_setup import bundle, resources, scripts
from vm_setup.resources import Clients, VmConfig

REGION = "us-east-1"        # même région que AWS/IotCore/setup_iot_core.ipynb
INSTANCE_TYPE = "t3.small"  # 2 Go : Analyzer (rejeu en mémoire) + Grafana + build Docker
PREFIX = "coffee-vm"

ROOT = Path.cwd().resolve().parents[1]   # racine du dépôt (le notebook est dans AWS/VM/)
STATE_FILE = Path.cwd() / ".state.json"  # identifiants des ressources, ignoré par git
assert (ROOT / "AWS/Grafana/docker-compose.local.yml").exists(), "lancer le notebook depuis AWS/VM/"

session = boto3.Session(region_name=REGION)
clients = Clients.from_session(session)
ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
cfg = VmConfig(region=REGION, account_id=ACCOUNT_ID, prefix=PREFIX, instance_type=INSTANCE_TYPE)
print("compte", ACCOUNT_ID, "| région", REGION, "| dépôt", ROOT)


def load_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state(**values):
    state = load_state() | values
    STATE_FILE.write_text(json.dumps(state, indent=2))
    return state

## 1. Prérequis

On vérifie, sans rien créer :
- qu'un VPC par défaut existe, car la VM s'y place et reçoit une IP publique pour joindre SSM, Docker Hub et GitHub en sortie ;
- que les fichiers à déployer sont présents, dont le jeu de rejeu ;
- que le plugin Session Manager est installé en local (il ne sert qu'au tunnel de l'étape 9).

In [ ]:
vpc_id = resources.default_vpc_id(clients.ec2)
files = bundle.bundle_files(ROOT)
plugin = shutil.which("session-manager-plugin")
print("VPC par défaut :", vpc_id)
print("fichiers à déployer :", len(files))
print("plugin Session Manager :", plugin or "ABSENT (étape 9) : brew install --cask session-manager-plugin")

## 2. Mot de passe administrateur de Grafana

Le mot de passe est généré une seule fois et stocké dans **SSM Parameter Store** (`SecureString`, chiffré, niveau standard gratuit). Il n'apparaît jamais dans ce notebook : la VM le lit elle-même au déploiement. Pour se connecter à Grafana :

```bash
aws ssm get-parameter --name /coffee-vm/grafana-admin-password --with-decryption --query Parameter.Value --output text --region us-east-1
```

(Le nom suit `PREFIX` : `/<PREFIX>/grafana-admin-password`.)

Grafana enregistre ce mot de passe à son **premier** démarrage, dans son volume. Le modifier ensuite dans Parameter Store n'a aucun effet (voir le README).

In [ ]:
created = resources.ensure_admin_password(clients.ssm, cfg)
print("paramètre", "créé" if created else "existant", ":", cfg.parameter_name)

## 3. Rôle IAM de la VM

La VM reçoit un rôle, via un *instance profile*, réduit au strict nécessaire :
- `AmazonSSMManagedInstanceCore` : l'agent SSM (Run Command et tunnel) ;
- lecture de **la seule** archive de déploiement dans S3 ;
- lecture du **seul** paramètre du mot de passe Grafana.

Les conteneurs n'héritent pas de ces droits, car IMDSv2 est limité à un saut réseau (étape 6).

In [ ]:
profile_arn = resources.ensure_instance_role(clients.iam, cfg)
save_state(instance_profile_arn=profile_arn)
print("instance profile :", profile_arn)

## 4. Security group

**Aucune règle entrante** : ni SSH ni port 3000. Si une règle entrante a été ajoutée à la main, elle est retirée. Toutes les sorties restent autorisées, pour l'agent SSM, les images Docker et le plugin Grafana Infinity.

In [ ]:
sg_id = resources.ensure_security_group(clients.ec2, cfg, vpc_id)
save_state(security_group_id=sg_id)
print("security group :", sg_id)

## 5. Archive de déploiement

Une liste blanche de fichiers est envoyée dans un bucket S3 privé, chiffré, à accès public bloqué et joignable uniquement en TLS : le code et le Dockerfile de l'Analyzer, le compose, la configuration et le dashboard Grafana, et le jeu de rejeu. Les chemins restent relatifs à la racine du dépôt, ce qui permet au compose de fonctionner tel quel sur la VM. Ni `.env`, ni venv, ni tests, ni certificats ne sont envoyés : un fichier qui ressemble à un secret dans ces dossiers bloque la création de l'archive. La somme SHA-256 de l'archive est enregistrée dans `.state.json`, et la VM la vérifie avant de déployer. Une archive remplacée dans S3 entre-temps est donc refusée.

**Relancer cette cellule, puis l'étape 8, redéploie la version courante du code.**

In [ ]:
resources.ensure_bucket(clients.s3, cfg)
data = bundle.build_bundle(ROOT)
sha256 = resources.upload_bundle(clients.s3, cfg, data)
save_state(bucket=cfg.bucket, bundle_key=cfg.bundle_key, bundle_sha256=sha256)
print(f"s3://{cfg.bucket}/{cfg.bundle_key} : {len(data) / 1e6:.1f} Mo, sha256 {sha256[:12]}…")

## 6. Instance EC2

- **Amazon Linux 2023**, dernière AMI de la région ;
- `t3.small` ;
- volume gp3 de 20 Go **chiffré** ;
- **IMDSv2 obligatoire** ;
- pas de paire de clés.

Au premier démarrage, le *user-data* installe Docker, puis les plugins `docker compose` et `buildx`, à des **versions épinglées** dont la somme SHA-256 publiée est vérifiée.

Une instance déjà créée est réutilisée (et redémarrée si elle est arrêtée). Le user-data ne s'exécute qu'à la création.

In [ ]:
state = load_state()
instance_id = resources.ensure_instance(
    clients.ec2,
    cfg,
    ami_id=resources.latest_ami(clients.ssm),
    security_group_id=state["security_group_id"],
    instance_profile_arn=state["instance_profile_arn"],
    user_data=scripts.user_data(),
)
save_state(instance_id=instance_id)
print("instance :", instance_id)

## 7. Attente de la VM

D'abord l'état `running`, puis l'agent SSM `Online`. Pour une nouvelle instance, cela prend 1 à 3 minutes.

In [ ]:
instance_id = load_state()["instance_id"]
resources.wait_running(clients.ec2, instance_id)
resources.wait_ssm_online(clients.ssm, instance_id)
print(instance_id, ": running, agent SSM online")

## 8. Déploiement

Le script ci-dessous est exécuté sur la VM par **SSM Run Command** :
1. attente de la fin du premier démarrage. S'il a échoué, la cellule s'arrête et affiche la fin de `/var/log/cloud-init-output.log` ;
2. téléchargement de l'archive, **vérification de sa somme SHA-256**, puis décompression dans un dossier de préparation ;
3. écriture de `AWS/Grafana/.env`, avec des droits réservés à root, à partir du mot de passe lu dans Parameter Store ;
4. synchronisation vers `/opt/coffee/app`. En cas d'échec avant cette étape, l'application en service n'est pas touchée ;
5. `docker compose -f docker-compose.local.yml up -d --build`, puis Grafana est recréé pour recharger sa configuration.

Le premier déploiement construit l'image de l'Analyzer, télécharge Grafana et installe le plugin Infinity : compter quelques minutes.

`REPLAY_RESTART = True` relance le rejeu depuis le début, utile quand les 8 h de données ont été rejouées et que toutes les machines sont passées hors ligne.

In [ ]:
REPLAY_RESTART = False

state = load_state()
commands = scripts.deploy_commands(
    cfg.bucket, cfg.bundle_key, state["bundle_sha256"], ACCOUNT_ID, cfg.parameter_name, REGION,
    restart_replay=REPLAY_RESTART,
)
output = resources.run_command(clients.ssm, state["instance_id"], commands, timeout_s=1800)
print(output[-3000:] or "déployé")

## 9. Vérification et accès à Grafana

On attend que les conteneurs `analyser` et `grafana` soient `healthy` et que Grafana réponde sur la VM. Ensuite, dans un **terminal**, on ouvre le tunnel avec la commande affichée. Elle reste active tant que le tunnel est ouvert.

Puis on ouvre http://localhost:3000, avec l'utilisateur `admin` et le mot de passe de l'étape 2. Le dashboard **Machines à café — flotte** est la page d'accueil.

In [ ]:
instance_id = load_state()["instance_id"]
status = resources.wait_healthy(clients.ssm, instance_id)
print("\nprêt :", status)
print("\ntunnel (dans un terminal) :")
print(f"  aws ssm start-session --region {REGION} --target {instance_id} "
      "--document-name AWS-StartPortForwardingSession --parameters portNumber=3000,localPortNumber=3000")
print("puis http://localhost:3000")

## 10. Arrêter / redémarrer la VM

Allumée en continu, la VM coûte environ 17 USD/mois (environ 15 USD pour l'instance et 1,6 USD pour le disque). Arrêtée, elle ne coûte plus que son disque. Au redémarrage, Docker relance les conteneurs (`restart: unless-stopped`), et le rejeu repart du début. Relancer ensuite les étapes 7 et 9.

In [ ]:
STOP = False
START = False

instance_id = load_state()["instance_id"]
if STOP:
    resources.set_running(clients.ec2, instance_id, running=False)
    print("arrêt demandé :", instance_id)
elif START:
    resources.set_running(clients.ec2, instance_id, running=True)
    print("démarrage demandé :", instance_id, "(relancer les étapes 7 et 9)")
else:
    print("STOP et START à False : rien à faire")

## 11. Nettoyage (optionnel)

Supprime **tout** : l'instance et son disque, le security group, le rôle et l'instance profile, le bucket et l'archive, le paramètre du mot de passe, puis `.state.json`. Mettre `CLEANUP = True` pour l'exécuter.

In [ ]:
CLEANUP = False

if CLEANUP:
    removed = resources.cleanup(clients, cfg)
    STATE_FILE.unlink(missing_ok=True)
    print("supprimé :", *removed, sep="\n  ") if removed else print("rien à supprimer")
else:
    print("CLEANUP = False : rien n'est supprimé")